In [ ]:
# 03c - Gradient Boosting (Kayla)

Baseline mean prediction and HistGradientBoostingRegressor tuned on the 2024 validation set.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

from shared_preprocessing import load_splits, make_tree_encoder

# the splits
X_train, y_train, X_val, y_val, X_test, y_test = load_splits("../data/stage2_outputs")

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (276685, 81) | y_train shape: (276685,)
X_val shape:   (30545, 81)   | y_val shape:   (30545,)
X_test shape:  (31535, 81)  | y_test shape:  (31535,)


In [9]:
def evaluate_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {name} ---")
    print(f"MAE:  ${mae:,.2f}")
    print(f"RMSE: ${rmse:,.2f}")
    print(f"R2:   {r2:.4f}\n")
    
    return {"model": name, "mae": mae, "rmse": rmse, "r2": r2}

In [10]:
# simple mean baseline
train_mean = y_train.mean()
print(f"Train mean price: ${train_mean:,.2f}\n")

val_baseline_preds = np.full_like(y_val, fill_value=train_mean)
baseline_val_res = evaluate_predictions("Mean Baseline (Val 2024)", y_val, val_baseline_preds)

Train mean price: $350,713.99

--- Mean Baseline (Val 2024) ---
MAE:  $227,823.97
RMSE: $444,016.55
R2:   -0.0463



In [11]:
gb_pipe = make_pipeline(
    make_tree_encoder(),
    HistGradientBoostingRegressor(random_state=42)
)

gb_pipe.fit(X_train, y_train)

val_preds_default = gb_pipe.predict(X_val)
gb_default_res = evaluate_predictions("HistGB - Default (Val 2024)", y_val, val_preds_default)

--- HistGB - Default (Val 2024) ---
MAE:  $114,852.25
RMSE: $196,645.88
R2:   0.7948



In [12]:
tuning_configs = [
    {"learning_rate": 0.05, "max_iter": 200, "max_depth": 8},
    {"learning_rate": 0.1,  "max_iter": 250, "max_depth": 10},
    {"learning_rate": 0.08, "max_iter": 300, "max_depth": 12},
]

best_score = -float("inf")
best_model = None
best_params = None

for cfg in tuning_configs:
    pipe = make_pipeline(
        make_tree_encoder(),
        HistGradientBoostingRegressor(
            learning_rate=cfg["learning_rate"],
            max_iter=cfg["max_iter"],
            max_depth=cfg["max_depth"],
            random_state=42
        )
    )
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    r2 = r2_score(y_val, preds)
    mae = mean_absolute_error(y_val, preds)
    print(f"params: {cfg} -> Val R2: {r2:.4f}, MAE: ${mae:,.2f}")
    
    if r2 > best_score:
        best_score = r2
        best_model = pipe
        best_params = cfg

print(f"\nBest config: {best_params} (Val R2 = {best_score:.4f})")

params: {'learning_rate': 0.05, 'max_iter': 200, 'max_depth': 8} -> Val R2: 0.7931, MAE: $114,757.63
params: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} -> Val R2: 0.8162, MAE: $107,526.22
params: {'learning_rate': 0.08, 'max_iter': 300, 'max_depth': 12} -> Val R2: 0.8142, MAE: $108,344.73

Best config: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} (Val R2 = 0.8162)


In [13]:
# Final evaluation on 2025 test set
test_preds = best_model.predict(X_test)
gb_test_res = evaluate_predictions("HistGB (Test 2025)", y_test, test_preds)

test_baseline_preds = np.full_like(y_test, fill_value=train_mean)
baseline_test_res = evaluate_predictions("Mean Baseline (Test 2025)", y_test, test_baseline_preds)

--- HistGB (Test 2025) ---
MAE:  $125,945.35
RMSE: $260,565.96
R2:   0.7655

--- Mean Baseline (Test 2025) ---
MAE:  $249,127.97
RMSE: $551,842.59
R2:   -0.0518



In [ ]:
### Notes on results

- The mean baseline gets a negative R² (-0.0518 on 2025 test) because market prices shifted upward compared to the 2017–2023 training mean.
- The tuned HistGradientBoosting model (`lr=0.1`, `max_iter=250`, `max_depth=10`) performed best on the validation set ($R^2 = 0.8162$). Going deeper to `max_depth=12` slightly reduced the score ($0.8142$), likely beginning to overfit.
- On the unseen 2025 test set, the final model achieved $R^2 = 0.7655$ and cut the baseline MAE roughly in half (from ~$249k down to ~$126k).
- The drop from validation (0.816) to test (0.765) is reasonable given the chronological split, reflecting price drift and inflation between 2024 and 2025.